# PhenoRewire — 2D Network Annotation Expansion Tutorial

This notebook demonstrates the `--run-2dnetwork` feature, which expands the PhenoRewire
priority features into a chemical-similarity (molecular) network to find annotated neighbours.

**What you will learn:**
1. What the 2D network expansion does and when to use it
2. How to run it via the Python API (same logic as `--run-2dnetwork` CLI flag)
3. How to interpret `annotation_summary.csv` and `annotation_expansion.csv`

**Prerequisites:** Run `docs/tutorials/tutorial.ipynb` first, or execute the cells in Section 1
below which reproduce the required pipeline outputs.

## 0. Setup

In [ ]:
from __future__ import annotations
import sys
import tempfile
from pathlib import Path

# Locate repo root — works whether the notebook is opened from docs/tutorials/ or repo root
_here = Path().resolve()
REPO_ROOT = _here
while not (REPO_ROOT / "pyproject.toml").exists() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import pandas as pd

print("Python:", sys.version)
print("Pandas:", pd.__version__)

## 1. Run the main PhenoRewire pipeline

The 2D network expansion requires `priority_features.csv` from a previous pipeline run.
We reproduce it here in a temp directory using the example dataset.

In [ ]:
from phenorewire.config import LeanConfig
from phenorewire.run import run

EXAMPLE_DIR = REPO_ROOT / "data"
features = pd.read_csv(EXAMPLE_DIR / "feature_table.csv")
metadata = pd.read_csv(EXAMPLE_DIR / "metadata.csv")

tmp = Path(tempfile.mkdtemp())
feat_path = tmp / "feature_table.csv"
meta_path = tmp / "metadata.csv"
outdir = tmp / "output"
outdir.mkdir()

features.to_csv(feat_path, index=False)
metadata.to_csv(meta_path, index=False)

config = LeanConfig(
    DATA_MATRIX=str(feat_path),
    METADATA=str(meta_path),
    OUTDIR=str(outdir),
    ANALYSIS_MODE="phenotype",
    GROUP_DEFINITION={
        "Reference": {"group_label": "reference"},
        "Case":      {"group_label": "case"},
    },
    PHENO_GROUP_REF="Reference",
    PHENO_GROUP_CASE="Case",
    FEATURE_ID_COL="feature_id",
    MZ_COL="mz",
    RT_COL="rt",
    NAME_COL="name",
    META_SAMPLE_COL="sample_id",
    INTENSITY_REGEX=r"^Sample_",
    N_PERMUTATIONS_PHENO=200,
    FDR_ALPHA=0.2,
    ADAPTIVE_SELECTION_THRESHOLDS=True,
    MIN_FEATURES_HARD_STOP=2,
    CORR_ABS_THRESHOLD=0.4,
    CORR_FDR_ALPHA=0.2,
    ADAPTIVE_NETWORK_THRESHOLDS=True,
    NETWORK_MIN_EDGES=2,
    NETWORK_MIN_EDGES_HARD_STOP=1,
    NETWORK_MIN_NODES_HARD_STOP=2,
    SIGN_SWITCH_MIN_R=0.4,
    LOUVAIN_STABILITY_CHECK=False,
    TRIAGE_TOPOLOGY_WEIGHT=0.7,
    TRIAGE_SELECTION_WEIGHT=0.3,
    LOG_TRANSFORM=True,
    NORM_METHOD="median",
    RANDOM_SEED=42,
)

run(config)
print("Main pipeline complete.")

## 2. What is the 2D network expansion?

A **molecular network** (e.g. from SpecReboot or GNPS) connects features by spectral cosine
similarity. Two metabolites are linked when their MS2 fragmentation spectra are similar,
suggesting they belong to the same chemical class.

The 2D network expansion answers: **"Which annotated molecules are chemically close to my
priority features?"**

```
PhenoRewire priority feature  →  k-hop BFS in molecular network  →  annotated neighbours
```

This is useful when a priority feature is unannotated: its chemical neighbours in the
molecular network often share a known class, enabling putative annotation.

## 3. Examine the synthetic molecular networks

The demo molecular network is in `data/example_molecular_network.graphml`.
This file is a small synthetic chemical-similarity graph whose node IDs match `data/feature_table.csv`.

In [ ]:
try:
    import networkx as nx
    HAS_NX = True
except ImportError:
    HAS_NX = False
    print("networkx not installed. Run: pip install networkx")

sr_path = REPO_ROOT / "data" / "example_molecular_network.graphml"

if HAS_NX and sr_path.exists():
    G_sr = nx.read_graphml(str(sr_path))
    print(f"Demo molecular network: {G_sr.number_of_nodes()} nodes, {G_sr.number_of_edges()} edges")
    annotated = [n for n, d in G_sr.nodes(data=True) if d.get("consensus_annotation")]
    print(f"Annotated nodes: {len(annotated)}")
else:
    print("Demo molecular network not found or networkx unavailable — skipping graph inspection")

## 4. Run the 2D network annotation expansion

In [ ]:
from phenorewire.twodn.chemical_integration import build_annotation_expansion

# Gather priority features from the pipeline run
priority_dfs = []
for p in sorted(outdir.rglob("priority_features.csv")):
    df = pd.read_csv(p)
    priority_dfs.append(df)

if not priority_dfs:
    print("No priority_features.csv found — check pipeline output.")
else:
    # Use the first network's priority features
    priority_df = priority_dfs[0]
    print(f"Priority features ({len(priority_df)} total):")
    print(priority_df[["feature_id", "triage_score"]].head(8).to_string(index=False))

    # Rewiring nodes
    rewiring_path = next(outdir.rglob("priority_rewired_nodes.csv"), None)
    rewiring_df = pd.read_csv(rewiring_path) if rewiring_path else pd.DataFrame()

    # Run expansion
    twodn_outdir = outdir / "2dnetwork"
    twodn_outdir.mkdir(exist_ok=True)

    if sr_path.exists() and pr_path.exists():
        summary_df, expansion_df = build_annotation_expansion(
            phenorewire_graphml=str(pr_path),
            specreboot_graphml=str(sr_path),
            priority_features_df=priority_df,
            rewiring_df=rewiring_df,
            top_n=10,
            hop_depth=2,
            min_cosine=0.7,
            output_dir=twodn_outdir,
        )
        print(f"\n2D network expansion complete.")
        print(f"  annotation_summary rows: {len(summary_df)}")
        print(f"  annotation_expansion rows: {len(expansion_df)}")
    else:
        print("GraphML files not found — skipping expansion.")

## 5. Interpret `annotation_summary.csv`

Each row summarises the chemical neighbourhood of one priority feature.

| Column | Meaning |
|---|---|
| `feature_id` | PhenoRewire priority feature |
| `n_mn_neighbors` | Number of features reachable within `hop_depth` hops |
| `n_annotated_neighbors` | How many neighbours have a known annotation |
| `annotation_confidence` | Fraction annotated (0 = isolated or all unknown; 1 = all annotated) |
| `top_annotation` | Annotation of the closest/best-scoring neighbour |
| `triage_score` | Score from PhenoRewire (passed through) |
| `rewiring_score` | Rewiring score from PhenoRewire (passed through) |

In [ ]:
if 'summary_df' in dir() and not summary_df.empty:
    print("Annotation summary (top features):")
    cols = [c for c in ["feature_id", "n_mn_neighbors", "n_annotated_neighbors", "annotation_confidence", "top_annotation"] if c in summary_df.columns]
    print(summary_df[cols].to_string(index=False))
else:
    print("No summary available — run section 4 first.")

**Reading annotation_confidence:**
- `0.0` — the feature has no neighbours in the molecular network, or all neighbours are unannotated.
  Consider it chemically isolated at this cosine threshold.
- `0.3–0.7` — partial annotation: some neighbours are known, others are not.
  The feature is likely in a partially characterised chemical class.
- `>= 0.8` — most neighbours are annotated. The feature probably belongs to a well-characterised class;
  `top_annotation` gives a strong putative class assignment.

## 6. Interpret `annotation_expansion.csv`

Each row is one (feature, neighbour) pair — the full hop-by-hop expansion table.

In [ ]:
if 'expansion_df' in dir() and not expansion_df.empty:
    print(f"Expansion table: {len(expansion_df)} rows")
    cols = [c for c in ["feature_id", "neighbor_id", "hop_distance", "edge_cosine", "neighbor_annotation"] if c in expansion_df.columns]
    print(expansion_df[cols].head(15).to_string(index=False))
else:
    print("No expansion data available — run section 4 first.")

**Column guide:**

| Column | Meaning |
|---|---|
| `feature_id` | The PhenoRewire priority feature |
| `neighbor_id` | A feature reachable from it in the molecular network |
| `hop_distance` | Number of edges in the shortest path (1 = direct neighbour) |
| `edge_cosine` | Minimum edge cosine along the path (lower = weaker chemical similarity) |
| `neighbor_annotation` | Annotation of the neighbour (or empty if unknown) |

**Tip:** Filter to `hop_distance == 1` and sort by `edge_cosine` descending to find the most
chemically similar annotated neighbours.

## 7. Per-feature subnetworks

Each priority feature also gets a per-feature subnetwork GraphML file in
`2dnetwork/subnetworks/`. Open these in Cytoscape to visualise the chemical neighbourhood.

In [ ]:
sub_dir = twodn_outdir / "subnetworks" if 'twodn_outdir' in dir() else None
if sub_dir and sub_dir.exists():
    graphml_files = sorted(sub_dir.glob("*.graphml"))
    print(f"Per-feature subnetworks ({len(graphml_files)} files):")
    for f in graphml_files:
        print(f"  {f.name}")
else:
    print("No subnetworks directory found — run section 4 first.")

## 8. Configuration parameters

The 2D network expansion is controlled by these parameters in your YAML config:

| Parameter | Default | Effect |
|---|---|---|
| `SPECREBOOT_NETWORK` | `null` | Path to SpecReboot/GNPS molecular network GraphML |
| `PHENOREWIRE_NETWORK` | `null` | Path to a saved PhenoRewire rewiring network GraphML |
| `MN_HOP_DEPTH` | `2` | Maximum hops from each priority feature |
| `MN_MIN_COSINE` | `0.70` | Minimum cosine similarity to traverse an edge |
| `ANNOTATION_TOP_N` | `20` | Number of priority features to expand |

**Guidelines:**
- Lower `MN_MIN_COSINE` (e.g. 0.5) to include more distant chemical relatives — useful for sparse networks.
- Lower `MN_HOP_DEPTH` to 1 for direct neighbours only — reduces noise in large networks.
- Increase `ANNOTATION_TOP_N` to expand more features.

## 9. CLI equivalent

Everything above is also available via the command line:

```bash
# Add network paths to your config.yaml:
# The bundled demo already has SPECREBOOT_NETWORK set:
phenorewire --config config_phenotype.yaml --run-2dnetwork
phenorewire --config config_phenotype.yaml --plot-networks
```

Outputs land in `OUTDIR/2dnetwork/` alongside the main results.